In [ ]:
# Example 1: Hatchling build hook entry point — verify the plugin registers via
# the hookimpl. hookimpl + the registration function live in hatch_cython.hooks;
# the hook class lives in hatch_cython.plugin.
from hatch_cython.hooks import hatch_register_build_hook, hookimpl
from hatch_cython.plugin import CythonBuildHook

print(f"hookimpl callable: {callable(hookimpl)}")
print(f"CythonBuildHook: {CythonBuildHook}")
print(f"PLUGIN_NAME = {CythonBuildHook.PLUGIN_NAME}")

# The hook class must declare a PLUGIN_NAME for Hatch to find it, and the
# hookimpl-decorated registration function must return the class.
assert CythonBuildHook.PLUGIN_NAME == "cython"
assert hatch_register_build_hook() is CythonBuildHook

In [ ]:
# Example 2: Config model — parse a minimal user config dict into the typed BuildConfig.
from hatch_cython.config.config import parse_from_dict, Config

class FakeMeta:
    def __init__(self, config):
        self.config = config

raw = {
    'src': 'src',
    'includes': [],
    'compile_args': [],
    'compile_kwargs': {},
}
cfg = parse_from_dict(FakeMeta(raw))
print(f"Config type: {type(cfg).__name__}")
print(f"src = {cfg.src}")
assert isinstance(cfg, Config)


In [ ]:
# Example 3: Platform constants — sanity check the OS detection helpers exist + return bools.
from hatch_cython import constants

names = [n for n in dir(constants) if not n.startswith('_')]
print(f"constants exports: {names[:20]}")

# Common platform helper names: MUST_UNIQUE, EXIST_TRIM, NORM, etc.
# Just verify the module loaded with real attributes.
assert len(names) > 0


In [ ]:
# Example 4: Types module — CompileArgs / Autoimport / etc. dataclasses should import + instantiate.
from hatch_cython.config import flags as flags_mod
from hatch_cython.config import includes as includes_mod

print(f"flags module: {flags_mod.__name__}")
print(f"flags exports: {[n for n in dir(flags_mod) if not n.startswith('_')][:15]}")
print(f"includes module: {includes_mod.__name__}")
print(f"includes exports: {[n for n in dir(includes_mod) if not n.startswith('_')][:15]}")

assert flags_mod is not None and includes_mod is not None


In [ ]:
# Example 5: Verify hatch_cython advertises itself as a Hatch build-hook plugin (entry point name).
import importlib.metadata as md

dist = md.distribution('hatch_cython')
print(f"distribution name: {dist.metadata['Name']}")
print(f"version: {dist.version}")

eps = dist.entry_points
build_hook_eps = [ep for ep in eps if 'hatch' in ep.group]
print(f"hatch entry points ({len(build_hook_eps)}):")
for ep in build_hook_eps:
    print(f"  group={ep.group}  name={ep.name}  value={ep.value}")

assert len(build_hook_eps) >= 1, "hatch_cython must register at least one hatch entry point"
